In [ ]:
import os

# import matplotlib.colors as colors
# import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick  # 1. Import the ticker module

# import matplotlib.gridspec as gridspec
# import numpy as np
import pandas as pd
import seaborn as sns

# from matplotlib.lines import Line2D
from matplotlib.patches import Patch

In [ ]:
data_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results/'
save_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results_figures/'

In [ ]:
# Order of all possible bins for plotting
order = ['(10, 20]', '(20, 30]', '(30, 40]', '(40, 50]',
         '(50, 60]', '(60, 70]', '(70, 80]', '(80, 90]',
         '(90, 100]',
         '(100, 150]', '(150, 200]', '(200, 500]']

In [ ]:
# filtered_order = order[1:]

In [ ]:
# pct_long['Bin'] = pd.Categorical(pct_long['Bin'], categories=order, ordered=True)

### Combined plot

In [ ]:
df = pd.read_parquet(os.path.join(data_dir, 'bennitt_wse_quality.parquet'))

In [ ]:
df.count_unfil.sum()

In [ ]:
df.columns

In [ ]:
df['total'] = 100
df['useable'] = ((df['count_useable'] / df['count_unfil']) * 100).round(2)
df['heights'] = ((df['count_heights'] / df['count_unfil']) * 100).round(2)
df['quality'] = ((df['count_high_qual'] / df['count_unfil']) * 100).round(2)

In [ ]:
df['Bin'] = pd.Categorical(df['Bin'], categories=order, ordered=True)

In [ ]:
df

In [ ]:
df[df['Bin'].isin(order[4:])]['count_useable'].sum() / df[df['Bin'].isin(order[4:])]['count_unfil'].sum()

In [ ]:
df[df['Bin'].isin(order[4:])]['count_heights'].sum() / df[df['Bin'].isin(order[4:])]['count_unfil'].sum()

In [ ]:
df[df['Bin'].isin(order[4:])]['count_high_qual'].sum() / df[df['Bin'].isin(order[4:])]['count_unfil'].sum()

In [ ]:
fig, ax = plt.subplots(figsize=(19,9))

plot_df = df#[df['Bin'].isin(order[1:])]

palette = ['silver',
           sns.color_palette('tab20b')[2],
           '#8DAE91',
           sns.color_palette('tab20b')[-3]]

sns.set_style('whitegrid')

sns.barplot(ax=ax, data=df,
            x='Bin', y='total',
            color='silver', alpha=0.5,
            legend=False, order=order)#[:-1])

sns.barplot(ax=ax, data=df,
            x='Bin', y='useable',
            color=sns.color_palette('tab20b')[2], alpha=0.5,
            legend=False, order=order)#[:-1])

sns.barplot(ax=ax, data=df,
            x='Bin', y='heights',
            color='#8DAE91', alpha=0.5,
            legend=False, order=order)#[:-1])

sns.barplot(ax=ax, data=df,
            x='Bin', y='quality',
            color=sns.color_palette('tab20b')[-3],
            legend=False, order=order)#[:-1])


# Axis labels and formatting
plt.xlabel('River width (m)', fontsize=18, fontweight='bold')
plt.ylabel('Percentage of reaches (%)', fontsize=18, fontweight='bold')
plt.xticks(rotation=-45, fontsize=16, fontweight='semibold')
plt.yticks(fontsize=16, fontweight='semibold')

# 2. Format y-axis ticks to display as percentages
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=100))

# Create a mapping of Bin to total_unique_reaches
counts_map = df.set_index('Bin')['count_unfil'].to_dict()

# Place text just above the 100% line
y_pos = 101 

for i, bin_name in enumerate(order):#[:-1]):
    if bin_name in counts_map:
        count_val = counts_map[bin_name]
        
        if not pd.isna(count_val):
            ax.text(i, y_pos, f'{int(count_val):,}', 
                    ha='center', va='bottom', fontsize=16, 
                    fontweight='bold', color='dimgray')

# Legend setup
legend_labels = ['Insufficient data',
                 'Unacceptable bias',
                 'Non-sensible timeseries',
                 'High-quality SWOT WSE']

handles = [
    Patch(facecolor=color, label=label, alpha=0.5 if idx < 3 else 1.0, edgecolor='none') 
    for idx, (color, label) in enumerate(zip(palette, legend_labels))
]
plt.legend(handles, legend_labels,
           # title='Region',
           framealpha=0.95,
           fontsize=16, title_fontsize=18, loc='lower right')

# Enable only horizontal grid lines
ax.yaxis.grid(True)
# ax.xaxis.grid(False)

# Optional: ensure they are behind the bars
ax.set_axisbelow(True)
# ax.set_yscale('log')

plt.ylim(0,100)

plt.savefig(f'{save_dir}quality.png', dpi=300, bbox_inches='tight')

In [ ]:
# 1. Melt the dataframe so all categories share a single column
df_melted = df.melt(
    id_vars=['Bin', 'count_unfil'], 
    value_vars=['total', 'useable', 'heights', 'quality'],
    var_name='Metric', 
    value_name='Percentage'
)

fig, ax = plt.subplots(figsize=(19,9))
sns.set_style('whitegrid')

palette = [sns.color_palette('tab20b')[3],
           sns.color_palette('tab20b')[2],
           '#8DAE91',
           sns.color_palette('tab20b')[-2]]

# 2. Plot everything in ONE call. Seaborn will automatically place bars side-by-side.
sns.barplot(
    ax=ax, 
    data=df_melted,
    x='Bin', 
    y='Percentage',
    hue='Metric',
    palette=palette,
    order=order
)

# 3. Text counts logic (remains clean)
counts_map = df.set_index('Bin')['count_unfil'].to_dict()
y_pos = 102 

for i, bin_name in enumerate(order):
    if bin_name in counts_map:
        count_val = counts_map[bin_name]
        if not pd.isna(count_val):
            ax.text(i, y_pos, f'{int(count_val):,}', 
                    ha='center', va='bottom', fontsize=11, 
                    fontweight='bold', color='black')

# 4. REMOVE the duplicate plt.ylim(0,100) from the bottom of your script!
plt.ylim(0, 100) 

# Adjust legend to look clean with the new hue
plt.legend(title='Metrics', framealpha=0.95, fontsize=16, title_fontsize=18, loc='lower right')
ax.yaxis.grid(True)
ax.set_axisbelow(True)